In [8]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, r2_score
from typing import Tuple, List, Any
from lightgbm import LGBMRegressor

import warnings, os
import mysql.connector as mysql
warnings.filterwarnings('ignore')
username = os.environ['MYSQL_user']
password = os.environ['MYSQL_password']
DB = mysql.connect(host = "localhost", user = username, passwd = password, database = "AIRBNB")
cursor = DB.cursor(buffered=True)
SEED = 17
compare_metric_name = 'RMSE'

## Utils

In [9]:
def read_table_from_db(table_name):
    df = pd.read_sql(f'SELECT * FROM {table_name}', con=DB)
    for col in df.columns:
        if len(df[col].unique()) == 2 or (df[col].dtype == 'object' and len(df[col].unique()) < 10):
            df[col] = df[col].astype('category')
    return df

In [10]:
def perform_cv(X: pd.DataFrame, y: pd.Series, algorithm: Any, cv: sklearn.model_selection = KFold(n_splits=5, shuffle=True, random_state=SEED), metric: sklearn.metrics = root_mean_squared_error) -> Tuple[List[float], List[float]]:
    """
    Perform cross-validation and return list of scores
    
    Args:
        X (pd.DataFrame): input data
        y (pd.Series): target data
        algorithm (Any): algorithm to use for training and prediction
        cv (sklearn.model_selection, default=KFold(n_splits=5, shuffle=True, random_state=SEED)): cross-validation strategy
        metric (sklearn.metrics, default=root_mean_squared_error): metric to use for evaluation
    
    Returns:
        Tuple[List[float], List[float]]: Tuple of lists of train and validation scores
    """
    train_scores, validation_scores = [], []
    for train_idx, val_idx in cv.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        algorithm.fit(X_train, y_train)
        y_train_pred = algorithm.predict(X_train)
        y_val_pred = algorithm.predict(X_val)
        train_scores.append(metric(y_train, y_train_pred))
        validation_scores.append(metric(y_val, y_val_pred))
    return train_scores, validation_scores

def evaluation(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series, algorithm: Any, metric: sklearn.metrics = root_mean_squared_error) -> Tuple[float, float, np.ndarray]:
    """
    Train the algorithm on the train data and evaluate on the train and test data
    
    Args:
        X_train (pd.DataFrame): input train data
        y_train (pd.Series): target train data
        X_test (pd.DataFrame): input test data
        y_test (pd.Series): target test data
        algorithm (Any): algorithm to use for training and prediction
        metric (sklearn.metrics, default=root_mean_squared_error): metric to use for evaluation
    
    Returns:
        Tuple[float, float, np.ndarray]: train_score, test_score, predictions on test data
    """
    algorithm.fit(X_train, y_train)
    y_train_pred = algorithm.predict(X_train)
    y_test_pred = algorithm.predict(X_test)
    train_results = metric(y_train, y_train_pred)
    test_results = metric(y_test, y_test_pred)
    return train_results, test_results, y_test_pred

# Load dataset

In [11]:
data = read_table_from_db('airbnb_data')
target_feature = 'log_price'
image_features = ["number_of_images", "number_of_bedroom_images", "number_of_bathroom_images", "number_of_living_room_images", "number_of_kitchen_images", "number_of_dining_room_images", "number_of_outside_building_images", "number_of_urban_environment_images", "number_of_other_images", "avg_warmth_hue", "avg_saturation", "avg_brightness", "avg_contrast_brightness", "avg_clarity"]
data = data[image_features + [target_feature]]
data

,number_of_images,number_of_bedroom_images,number_of_bathroom_images,number_of_living_room_images,number_of_kitchen_images,number_of_dining_room_images,number_of_outside_building_images,number_of_urban_environment_images,number_of_other_images,avg_warmth_hue,avg_saturation,avg_brightness,avg_contrast_brightness,avg_clarity,log_price
0,19,3,1,5,3,4,0,2,1,0.858955,0.347937,0.612404,0.219364,0.406733,5.48064
1,8,3,1,0,1,3,0,0,0,0.786527,0.335939,0.702361,0.197192,0.561365,4.27667
2,15,1,6,1,1,5,1,0,0,0.766307,0.196585,0.797081,0.166314,0.807861,4.39445
3,8,3,1,2,0,0,1,1,0,0.692493,0.273260,0.516383,0.232102,0.250748,4.17439
4,32,16,3,4,3,1,0,1,4,0.890491,0.410061,0.510274,0.235629,0.264580,4.17439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19454,7,3,1,1,2,0,0,0,0,0.850956,0.143719,0.641540,0.210288,0.573581,4.06044
19455,13,2,1,3,3,2,2,0,0,0.870749,0.173025,0.624783,0.208991,0.461112,4.33073
19456,19,7,1,2,2,1,6,0,0,0.672730,0.259213,0.477982,0.221113,0.244029,3.71357
19457,7,3,3,1,0,0,0,0,0,0.715691,0.217306,0.513331,0.242898,0.273807,4.75359


## Split dataset

In [12]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=SEED)

## Base model

In [13]:
model = LGBMRegressor(random_state=SEED, verbose=-1, linear_tree=True)
train_scores, validation_scores = perform_cv(train_data[image_features], train_data[target_feature], model, cv=KFold(n_splits=5, shuffle=True, random_state=SEED), metric=root_mean_squared_error)
print(f"Train {compare_metric_name}: {np.mean(train_scores):.4f} +- {np.std(train_scores):.4f}")
print(f"Validation {compare_metric_name}: {np.mean(validation_scores):.4f} +- {np.std(validation_scores):.4f}")

Train RMSE: 0.5064 +- 0.0022
Validation RMSE: 0.6233 +- 0.0060


In [14]:
X_train = train_data[image_features]
y_train = train_data[target_feature]
X_test = test_data[image_features]
y_test = test_data[target_feature]
model = LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1, objective='regression', metric=compare_metric_name, linear_tree=True)
model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
metrics = {
    'R2': r2_score,
    'R2_adj': lambda y_true, y_pred: 1 - (1 - r2_score(y_true, y_pred)) * (len(y_true) - 1) / (len(y_true) - X_train.shape[1] - 1),
    'MSE': mean_squared_error,
    'RMSE': root_mean_squared_error,
    'MAE': mean_absolute_error,
}
results = {}
for metric_name, metric in metrics.items():
    train_score = metric(y_train, y_train_pred)
    test_score = metric(y_test, y_test_pred)
    results[metric_name] = {
        'train': train_score,
        'test': test_score
    }
df = pd.DataFrame(results).T
df

,train,test
R2,0.501429,0.305875
R2_adj,0.500980,0.303368
MSE,0.270795,0.391667
RMSE,0.520380,0.625833
MAE,0.412308,0.493416
